In [1]:
!pip install owlready2

     ---------------------------------------- 0.0/27.3 MB ? eta -:--:--
     ---------------------------------------- 0.0/27.3 MB ? eta -:--:--
     ---------------------------------------- 0.0/27.3 MB ? eta -:--:--
     --------------------------------------- 0.0/27.3 MB 217.9 kB/s eta 0:02:06
     --------------------------------------- 0.1/27.3 MB 328.2 kB/s eta 0:01:24
     --------------------------------------- 0.1/27.3 MB 438.1 kB/s eta 0:01:03
     --------------------------------------- 0.1/27.3 MB 532.5 kB/s eta 0:00:52
     --------------------------------------- 0.2/27.3 MB 737.3 kB/s eta 0:00:37
     --------------------------------------- 0.2/27.3 MB 737.3 kB/s eta 0:00:37
     --------------------------------------- 0.3/27.3 MB 952.6 kB/s eta 0:00:29
     --------------------------------------- 0.3/27.3 MB 952.6 kB/s eta 0:00:29
     --------------------------------------- 0.3/27.3 MB 952.6 kB/s eta 0:00:29
      -------------------------------------- 0.5/27.3 MB 994.4 k

In [91]:
from owlready2 import *
from pathlib import Path

cc_onto = get_ontology("http://climate-change.org/kg/swrl#")

with cc_onto:
    class Organization(Thing): pass
    class GeopoliticalEntity(Thing): pass
    class Agreement(Thing): pass
    class PolicyLinkedOrg(Organization): pass
    class locatedIn(Organization >> GeopoliticalEntity): pass
    class signedBy(Agreement >> GeopoliticalEntity): pass

FAMILY_PATH = Path(r"C:\Users\rache\OneDrive - De Vinci\web datamining and semantics\td\lab1\family.owl")

BASE = "http://www.owl-ontologies.com/unnamed.owl#"
AGE_PROP = BASE + "age"

onto = get_ontology("http://www.owl-ontologies.com/unnamed.owl")
with open(FAMILY_PATH, "rb") as f:
    onto.graph.parse(f)

print("Ontology loaded.")

# Read individuals and ages from the RDF triple store
individuals = {}
age_storid = onto.graph._abbreviate(AGE_PROP, create_if_missing=False)

for s, p, o, d in onto.graph.execute(
    "SELECT s, p, o, d FROM quads WHERE p=?", [age_storid]
):
    name = onto.graph._unabbreviate(s).replace(BASE, "")
    individuals[name] = int(o)

print(f"Individuals found: {len(individuals)}")
print()
print("SWRL rule:") #si une personne a un âge supérieur à 60, alors c’est une OldPerson
print("  Person(?p), age(?p, ?a), swrlb:greaterThan(?a, 60) -> OldPerson(?p)")
print()
print("Individuals evaluated:")

inferred = []
for name, age_val in sorted(individuals.items()):
    is_old = age_val > 60
    if is_old:
        inferred.append((name, age_val))
    flag = " --> inferred as OldPerson" if is_old else ""
    print(f"  {name} (age: {age_val}){flag}")

print()
print("OldPerson instances inferred (age > 60):")
for name, age_val in inferred:
    print(f"  {name} (age: {age_val})")

Ontology loaded.
Individuals found: 12

SWRL rule:
  Person(?p), age(?p, ?a), swrlb:greaterThan(?a, 60) -> OldPerson(?p)

Individuals evaluated:
  Alex (age: 25)
  Chloé (age: 18)
  Claude (age: 5)
  John (age: 45)
  Marie (age: 69) --> inferred as OldPerson
  Michael (age: 5)
  Paul (age: 38)
  Pedro (age: 10)
  Peter (age: 70) --> inferred as OldPerson
  Sylvie (age: 30)
  Thomas (age: 40)
  Tom (age: 10)

OldPerson instances inferred (age > 60):
  Marie (age: 69)
  Peter (age: 70)


In [93]:
# SWRL rule on the Climate Change KB (simulated reasoning)
# Rule: Organization(?o), locatedIn(?o, ?g), signedBy(?a, ?g) -> PolicyLinkedOrg(?o)
# an organization located in a country that signed a climate agreement is inferred as a PolicyLinkedOrg

print("SWRL rule:")
print("  Organization(?o), locatedIn(?o, ?g), signedBy(?a, ?g) -> PolicyLinkedOrg(?o)")
print()

# Knowledge base
organizations = {
    "IPCC":          "Switzerland",
    "UnitedNations": "UnitedStates",
    "NASA":          "UnitedStates",
    "NOAA":          "UnitedStates",
    "FEMA":          "UnitedStates",
}

signatories = {
    "ParisAgreement": ["France", "UnitedStates", "UnitedKingdom", "Switzerland",
                       "Germany", "China", "India", "Brazil", "Canada", "Australia"],
    "KyotoProtocol":  ["France", "UnitedKingdom", "Germany", "China",
                       "India", "Canada", "Japan", "Russia"],
}

# Apply rule
policy_linked = []
for org, country in organizations.items():
    for agreement, countries in signatories.items():
        if country in countries:
            policy_linked.append((org, country, agreement))
            break

print("Reasoning result:")
print(f"  {'Organization':<20} {'LocatedIn':<20} {'Agreement':<20} {'Inferred'}")
print("  " + "-" * 75)
for org, country in organizations.items():
    inferred = any(p[0] == org for p in policy_linked)
    agreement = next((p[2] for p in policy_linked if p[0] == org), "none")
    print(f"  {org:<20} {country:<20} {agreement:<20} {'PolicyLinkedOrg' if inferred else ''}")

print()
print("PolicyLinkedOrg instances inferred:")
for org, country, agreement in policy_linked:
    print(f"  {org} (via {agreement}, located in {country})")

cc_onto.save(file=r"C:\Users\rache\OneDrive - De Vinci\web datamining and semantics\td\lab1\climate_swrl.owl", format="rdfxml")
print("climate_swrl.owl saved.")

SWRL rule:
  Organization(?o), locatedIn(?o, ?g), signedBy(?a, ?g) -> PolicyLinkedOrg(?o)

Reasoning result:
  Organization         LocatedIn            Agreement            Inferred
  ---------------------------------------------------------------------------
  IPCC                 Switzerland          ParisAgreement       PolicyLinkedOrg
  UnitedNations        UnitedStates         ParisAgreement       PolicyLinkedOrg
  NASA                 UnitedStates         ParisAgreement       PolicyLinkedOrg
  NOAA                 UnitedStates         ParisAgreement       PolicyLinkedOrg
  FEMA                 UnitedStates         ParisAgreement       PolicyLinkedOrg

PolicyLinkedOrg instances inferred:
  IPCC (via ParisAgreement, located in Switzerland)
  UnitedNations (via ParisAgreement, located in UnitedStates)
  NASA (via ParisAgreement, located in UnitedStates)
  NOAA (via ParisAgreement, located in UnitedStates)
  FEMA (via ParisAgreement, located in UnitedStates)
climate_swrl.owl saved.


### knowledge graph embeddings

In [75]:
!pip install pykeen

     ---------------------------------------- 0.0/85.9 kB ? eta -:--:--
     ---- ----------------------------------- 10.2/85.9 kB ? eta -:--:--
     --------- ---------------------------- 20.5/85.9 kB 217.9 kB/s eta 0:00:01
     ------------------ ------------------- 41.0/85.9 kB 281.8 kB/s eta 0:00:01
     --------------------------- ---------- 61.4/85.9 kB 363.1 kB/s eta 0:00:01
     -------------------------------------- 85.9/85.9 kB 404.5 kB/s eta 0:00:00
   ---------------------------------------- 0.0/730.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/730.3 kB ? eta -:--:--
   --- ------------------------------------ 71.7/730.3 kB ? eta -:--:--
   ------ --------------------------------- 122.9/730.3 kB 1.8 MB/s eta 0:00:01
   ---------- ----------------------------- 184.3/730.3 kB 1.9 MB/s eta 0:00:01
   ------------ --------------------------- 225.3/730.3 kB 1.7 MB/s eta 0:00:01
   ---------------- ----------------------- 307.2/730.3 kB 1.7 MB/s eta 0:00:01
 

In [95]:
#data prep + split train/test
import pandas as pd
import numpy as np
from pathlib import Path
from rdflib import Graph, URIRef, Literal

EXPANDED_PATH = Path(r"C:\Users\rache\OneDrive - De Vinci\web datamining and semantics\td\lab1\expanded_kb.nt")

print("Loading expanded KB...")
g = Graph()
g.parse(str(EXPANDED_PATH), format="nt")
print(f"Total triples loaded: {len(g)}")

# Keep only URI-URI-URI triples (no literals) for KGE
triples = []
for s, p, o in g:
    if isinstance(s, URIRef) and isinstance(p, URIRef) and isinstance(o, URIRef):
        triples.append((str(s), str(p), str(o)))

print(f"URI-only triples (usable for KGE): {len(triples)}")

# Shuffle and split 80/10/10
np.random.seed(42)
np.random.shuffle(triples)

n = len(triples)
n_train = int(0.8 * n)
n_valid = int(0.1 * n)

train = triples[:n_train]
valid = triples[n_train:n_train + n_valid]
test  = triples[n_train + n_valid:]

print(f"\nSplit:")
print(f"  Train: {len(train)}")
print(f"  Valid: {len(valid)}")
print(f"  Test:  {len(test)}")

# Save as tab-separated txt files
OUT_DIR = Path(r"C:\Users\rache\OneDrive - De Vinci\web datamining and semantics\td\lab1")

for split_name, split_data in [("train", train), ("valid", valid), ("test", test)]:
    path = OUT_DIR / f"{split_name}.txt"
    with open(path, "w", encoding="utf-8") as f:
        for s, p, o in split_data:
            f.write(f"{s}\t{p}\t{o}\n")
    print(f"  {split_name}.txt saved: {path}")

Loading expanded KB...
Total triples loaded: 70570
URI-only triples (usable for KGE): 52459

Split:
  Train: 41967
  Valid: 5245
  Test:  5247
  train.txt saved: C:\Users\rache\OneDrive - De Vinci\web datamining and semantics\td\lab1\train.txt
  valid.txt saved: C:\Users\rache\OneDrive - De Vinci\web datamining and semantics\td\lab1\valid.txt
  test.txt saved: C:\Users\rache\OneDrive - De Vinci\web datamining and semantics\td\lab1\test.txt


In [79]:
from pykeen.pipeline import pipeline
from pykeen.triples import TriplesFactory

OUT_DIR = Path(r"C:\Users\rache\OneDrive - De Vinci\web datamining and semantics\td\lab1")

# Load splits
tf_train = TriplesFactory.from_path(OUT_DIR / "train.txt")
tf_valid = TriplesFactory.from_path(
    OUT_DIR / "valid.txt",
    entity_to_id=tf_train.entity_to_id,
    relation_to_id=tf_train.relation_to_id,
)
tf_test = TriplesFactory.from_path(
    OUT_DIR / "test.txt",
    entity_to_id=tf_train.entity_to_id,
    relation_to_id=tf_train.relation_to_id,
)

print(f"Entities:  {tf_train.num_entities}")
print(f"Relations: {tf_train.num_relations}")
print(f"Train:     {tf_train.num_triples}")
print(f"Valid:     {tf_valid.num_triples}")
print(f"Test:      {tf_test.num_triples}")

print("\nTraining TransE...")
result_transe = pipeline(
    training=tf_train,
    validation=tf_valid,
    testing=tf_test,
    model="TransE",
    model_kwargs=dict(embedding_dim=100),
    optimizer="Adam",
    optimizer_kwargs=dict(lr=0.001),
    training_kwargs=dict(num_epochs=100, batch_size=512),
    negative_sampler="basic",
    evaluator_kwargs=dict(filtered=True),
    random_seed=42,
)

print("\nTransE results:")
metrics_transe = result_transe.metric_results.to_dict()
for metric in ["hits_at_1", "hits_at_3", "hits_at_10", "inverse_harmonic_mean_rank"]:
    val = metrics_transe.get("both.realistic." + metric, "N/A")
    label = "MRR" if metric == "inverse_harmonic_mean_rank" else metric
    print(f"  {label}: {val:.4f}" if isinstance(val, float) else f"  {label}: {val}")

You're trying to map triples with 4416 entities and 113 relations that are not in the training set. These triples will be excluded from the mapping.
In total 4424 from 5245 triples were filtered out
You're trying to map triples with 4399 entities and 115 relations that are not in the training set. These triples will be excluded from the mapping.
In total 4403 from 5247 triples were filtered out
No cuda devices were available. The model runs on CPU


Entities:  37020
Relations: 2593
Train:     41967
Valid:     821
Test:      844

Training TransE...


C:\Users\rache\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training epochs on cpu:   0%|          | 0/100 [00:00<?, ?epoch/s]

Training batches on cpu:   0%|          | 0.00/82.0 [00:00<?, ?batch/s]

...

Training batches on cpu:   0%|          | 0.00/82.0 [00:00<?, ?batch/s]

Training batches on cpu:   0%|          | 0.00/82.0 [00:00<?, ?batch/s]

Evaluating on cpu:   0%|          | 0.00/844 [00:00<?, ?triple/s]

INFO:pykeen.evaluation.evaluator:Evaluation took 11.39s seconds



TransE results:
  hits_at_1: N/A
  hits_at_3: N/A
  hits_at_10: N/A
  MRR: N/A


In [81]:
metrics = result_transe.metric_results.to_dict()
for k, v in list(metrics.items())[:30]:
    print(k, "->", v)

head -> {'optimistic': {'variance': 29182642.706502263, 'arithmetic_mean_rank': 2268.690758293839, 'adjusted_geometric_mean_rank_index': 0.9915191050809858, 'z_arithmetic_mean_rank': 44.15099254688705, 'adjusted_arithmetic_mean_rank_index': 0.8774460527081653, 'standard_deviation': 5402.096140064731, 'inverse_arithmetic_mean_rank': 0.0004407828596049144, 'inverse_geometric_mean_rank': 0.008580543279938787, 'z_geometric_mean_rank': 28.853594806271637, 'geometric_mean_rank': 116.54273714089743, 'z_inverse_harmonic_mean_rank': 452.8100466966747, 'count': 844.0, 'adjusted_arithmetic_mean_rank': 0.122601364985661, 'median_absolute_deviation': 151.9667273968242, 'inverse_harmonic_mean_rank': 0.10410686970927549, 'harmonic_mean_rank': 9.605514052939622, 'inverse_median_rank': 0.009569377990430622, 'adjusted_inverse_harmonic_mean_rank': 0.10383817468229188, 'median_rank': 104.5, 'hits_at_1': 0.01066350710900474, 'hits_at_3': 0.15165876777251186, 'hits_at_5': 0.1990521327014218, 'hits_at_10': 0

In [83]:
print("Training DistMult...")
result_distmult = pipeline(
    training=tf_train,
    validation=tf_valid,
    testing=tf_test,
    model="DistMult",
    model_kwargs=dict(embedding_dim=100),
    optimizer="Adam",
    optimizer_kwargs=dict(lr=0.001),
    training_kwargs=dict(num_epochs=100, batch_size=512),
    negative_sampler="basic",
    evaluator_kwargs=dict(filtered=True),
    random_seed=42,
)

# Extract metrics for both models
def get_metrics(result):
    r = result.metric_results.to_dict()["both"]["realistic"]
    return {
        "MRR":      round(r["inverse_harmonic_mean_rank"], 4),
        "Hits@1":   round(r["hits_at_1"], 4),
        "Hits@3":   round(r["hits_at_3"], 4),
        "Hits@10":  round(r["hits_at_10"], 4),
        "MeanRank": round(r["arithmetic_mean_rank"], 1),
    }

m_transe   = get_metrics(result_transe)
m_distmult = get_metrics(result_distmult)

print("\nModel Comparison (both head+tail, filtered, realistic)")
print(f"{'Metric':<12} {'TransE':>10} {'DistMult':>10}")
print("-" * 34)
for metric in ["MRR", "Hits@1", "Hits@3", "Hits@10", "MeanRank"]:
    print(f"{metric:<12} {m_transe[metric]:>10} {m_distmult[metric]:>10}")

INFO:pykeen.pipeline.api:Using device: None
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding(
  (regularizer): LpRegularizer()
)


Training DistMult...


Training epochs on cpu:   0%|          | 0/100 [00:00<?, ?epoch/s]

Training batches on cpu:   0%|          | 0.00/82.0 [00:00<?, ?batch/s]

Training batches on cpu:   0%|          | 0.00/82.0 [00:00<?, ?batch/s]

...

Training batches on cpu:   0%|          | 0.00/82.0 [00:00<?, ?batch/s]

Training batches on cpu:   0%|          | 0.00/82.0 [00:00<?, ?batch/s]

Evaluating on cpu:   0%|          | 0.00/844 [00:00<?, ?triple/s]

INFO:pykeen.evaluation.evaluator:Evaluation took 3.76s seconds



Model Comparison (both head+tail, filtered, realistic)
Metric           TransE   DistMult
----------------------------------
MRR              0.1014     0.1352
Hits@1           0.0148     0.0865
Hits@3           0.1434     0.1434
Hits@10          0.2565     0.2382
MeanRank         2752.8     7605.1


In [85]:
#sensiblite à la taille du kb
import numpy as np

def run_pipeline_on_subset(triples_list, size_label, model_name):
    """Train and evaluate a model on a subset of triples."""
    np.random.seed(42)
    subset = triples_list[:size_label]
    np.random.shuffle(subset)

    n = len(subset)
    n_train = int(0.8 * n)
    n_valid = int(0.1 * n)

    train_sub = subset[:n_train]
    valid_sub = subset[n_train:n_train + n_valid]
    test_sub  = subset[n_train + n_valid:]

    # Need at least a few triples in each split
    if len(train_sub) < 10 or len(valid_sub) < 2 or len(test_sub) < 2:
        return None

    tf_tr = TriplesFactory.from_labeled_triples(
        np.array(train_sub), create_inverse_triples=False
    )
    tf_va = TriplesFactory.from_labeled_triples(
        np.array(valid_sub),
        entity_to_id=tf_tr.entity_to_id,
        relation_to_id=tf_tr.relation_to_id,
        create_inverse_triples=False,
    )
    tf_te = TriplesFactory.from_labeled_triples(
        np.array(test_sub),
        entity_to_id=tf_tr.entity_to_id,
        relation_to_id=tf_tr.relation_to_id,
        create_inverse_triples=False,
    )

    result = pipeline(
        training=tf_tr,
        validation=tf_va,
        testing=tf_te,
        model=model_name,
        model_kwargs=dict(embedding_dim=100),
        optimizer="Adam",
        optimizer_kwargs=dict(lr=0.001),
        training_kwargs=dict(num_epochs=50, batch_size=256),
        negative_sampler="basic",
        evaluator_kwargs=dict(filtered=True),
        random_seed=42,
    )
    r = result.metric_results.to_dict()["both"]["realistic"]
    return {
        "MRR":     round(r["inverse_harmonic_mean_rank"], 4),
        "Hits@1":  round(r["hits_at_1"], 4),
        "Hits@10": round(r["hits_at_10"], 4),
    }

# Load all URI triples from expanded KB
from rdflib import Graph, URIRef
g_full = Graph()
g_full.parse(str(EXPANDED_PATH), format="nt")

all_triples = [
    [str(s), str(p), str(o)]
    for s, p, o in g_full
    if isinstance(s, URIRef) and isinstance(p, URIRef) and isinstance(o, URIRef)
]
np.random.seed(42)
np.random.shuffle(all_triples)

print(f"Total URI triples available: {len(all_triples)}")

sizes = {"20k": 20000, "50k": 50000, "full": len(all_triples)}
sensitivity_results = {}

for label, size in sizes.items():
    subset = all_triples[:size]
    print(f"\nRunning TransE on {label} ({len(subset)} triples)...")
    metrics = run_pipeline_on_subset(subset, size, "TransE")
    sensitivity_results[label] = metrics
    if metrics:
        print(f"  MRR={metrics['MRR']}  Hits@1={metrics['Hits@1']}  Hits@10={metrics['Hits@10']}")

print("\nSize Sensitivity Analysis (TransE)")
print(f"{'Size':<8} {'MRR':>8} {'Hits@1':>8} {'Hits@10':>8}")
print("-" * 36)
for label, metrics in sensitivity_results.items():
    if metrics:
        print(f"{label:<8} {metrics['MRR']:>8} {metrics['Hits@1']:>8} {metrics['Hits@10']:>8}")

Total URI triples available: 52459

Running TransE on 20k (20000 triples)...


INFO:pykeen.pipeline.api:Using device: None
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()


Training epochs on cpu:   0%|          | 0/50 [00:00<?, ?epoch/s]

Training batches on cpu:   0%|          | 0.00/63.0 [00:00<?, ?batch/s]

Training batches on cpu:   0%|          | 0.00/63.0 [00:00<?, ?batch/s]

...

Training batches on cpu:   0%|          | 0.00/164 [00:00<?, ?batch/s]

Training batches on cpu:   0%|          | 0.00/164 [00:00<?, ?batch/s]

Evaluating on cpu:   0%|          | 0.00/812 [00:00<?, ?triple/s]

INFO:pykeen.evaluation.evaluator:Evaluation took 10.40s seconds


  MRR=0.1072  Hits@1=0.0271  Hits@10=0.2488

Size Sensitivity Analysis (TransE)
Size          MRR   Hits@1  Hits@10
------------------------------------
20k        0.0281   0.0038   0.0679
50k         0.093   0.0161   0.2358
full       0.1072   0.0271   0.2488
